# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit: One row is one query for one content item and client over a 90-day window. I aggregate these query-level records to client-content level for feature engineering.

Features: 90-day impressions and clicks, last-30 and previous-30 performance, average positions, content-level impressions and query counts, rare-query measures, and anonymized-impression share.

Label: avg_position_90d.

Context: Client, content, and query IDs, plus the 90-day window dates.

Excluded: Entity IDs are excluded from model features because they are identifiers, not performance measures. client_hash_id is retained for grouped train/test splitting.

Verification: I checked the grain, unique clients/content/queries, missing values, date windows, and the HAVING COUNT(*) >= 10 threshold using DuckDB queries.

Limits: The data cannot explain causation, user intent, competitor actions, or algorithm changes. Client history is unbalanced, some early data may be GSC-only, and the 90-day and 30-day windows are related rather than independent.

In [2]:
import duckdb

con = duckdb.connect()

In [9]:
%whos

Variable   Type                  Data/Info
------------------------------------------
con        DuckDBPyConnection    <_duckdb.DuckDBPyConnecti<...>ct at 0x0000015755379730>
duckdb     module                <module 'duckdb' from 'c:<...>es\\duckdb\\__init__.py'>


In [5]:
con.execute("""
    SHOW TABLES
""").df()

,name


In [11]:
import os
import getpass

In [12]:
HF_TOKEN = getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

In [13]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [14]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [15]:
con.sql(f"""
    SELECT *
    FROM {TABLES['fact_query_90d']}
    LIMIT 5
""").df()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [16]:
# Check the unit of analysis
df = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_query_90d']}
    LIMIT 5
""").df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 5
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [17]:
# Check unique clients, content items, and queries
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT query_hash_id) AS queries
    FROM {TABLES['fact_query_90d']}
""").df()

,total_rows,clients,content_items,queries
0,2414248,52,133852,1180090


In [18]:
# Check the date window
con.sql(f"""
    SELECT
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end
    FROM {TABLES['fact_query_90d']}
""").df()

,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30


2. Fields: feature / label / context / excluded

Sort every field you plan to touch into these four buckets. Excluded needs a why.

Features:

impressions_90d
clicks_90d
impressions_last30
clicks_last30
impressions_prev30
clicks_prev30
avg_position_last30
avg_position_prev30
content_total_impressions_90d
content_visible_query_count
rare_query_count
rare_impressions_share
anonymized_impressions_share

Label:

avg_position_90d

Context:

client_hash_id
content_hash_id
query_hash_id
window_start
window_end

Excluded:

client_hash_id
content_hash_id
query_hash_id

These identifiers are excluded from the model features because they identify entities rather than represent search performance. However, client_hash_id is retained separately when using GroupShuffleSplit so that data from the same client does not appear in both training and testing sets.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
# Check missing values in important fields
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(client_hash_id) AS missing_client_id,
        COUNT(*) - COUNT(content_hash_id) AS missing_content_id,
        COUNT(*) - COUNT(query_hash_id) AS missing_query_id,
        COUNT(*) - COUNT(avg_position_90d) AS missing_avg_position,
        COUNT(*) - COUNT(impressions_90d) AS missing_impressions
    FROM {TABLES['fact_query_90d']}
""").df()

,total_rows,missing_client_id,missing_content_id,missing_query_id,missing_avg_position,missing_impressions
0,2414248,0,0,0,0,0


In [20]:
# Check how many query records exist per client-content pair
con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS query_count
    FROM {TABLES['fact_query_90d']}
    GROUP BY
        client_hash_id,
        content_hash_id
    ORDER BY query_count DESC
    LIMIT 10
""").df()

,client_hash_id,content_hash_id,query_count
0,client_e547b89c05043229,content_eadb33b5df496f4a,7889
1,client_e547b89c05043229,content_545bb6cc7081ded3,3378
2,client_e547b89c05043229,content_0e03de7680314cd5,2241
3,client_e547b89c05043229,content_9ef3d7516483e665,1577
4,client_23a62021009f63c4,content_661a7734f691bef5,1551
5,client_e547b89c05043229,content_8d7d99f109e19aa2,1399
6,client_23a62021009f63c4,content_93a9b8328d4fd032,1358
7,client_73cda7b4e4f265ea,content_33d31496fca9665e,1335
8,client_e547b89c05043229,content_77276ad7a26f4905,1294
9,client_e547b89c05043229,content_61215c724c8220ae,1285


In [21]:
# Check the 90-day performance fields
con.sql(f"""
    SELECT
        MIN(impressions_90d) AS min_impressions,
        MAX(impressions_90d) AS max_impressions,
        AVG(impressions_90d) AS avg_impressions,
        MIN(avg_position_90d) AS min_position,
        MAX(avg_position_90d) AS max_position,
        AVG(avg_position_90d) AS avg_position
    FROM {TABLES['fact_query_90d']}
""").df()

,min_impressions,max_impressions,avg_impressions,min_position,max_position,avg_position
0,10,543044,87.705578,0.0,432.75,19.767473


In [22]:
# Check client-content groups with at least 10 query records
con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS query_count
    FROM {TABLES['fact_query_90d']}
    GROUP BY
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) >= 10
    ORDER BY query_count DESC
    LIMIT 10
""").df()

,client_hash_id,content_hash_id,query_count
0,client_e547b89c05043229,content_eadb33b5df496f4a,7889
1,client_e547b89c05043229,content_545bb6cc7081ded3,3378
2,client_e547b89c05043229,content_0e03de7680314cd5,2241
3,client_e547b89c05043229,content_9ef3d7516483e665,1577
4,client_23a62021009f63c4,content_661a7734f691bef5,1551
5,client_e547b89c05043229,content_8d7d99f109e19aa2,1399
6,client_23a62021009f63c4,content_93a9b8328d4fd032,1358
7,client_73cda7b4e4f265ea,content_33d31496fca9665e,1335
8,client_e547b89c05043229,content_77276ad7a26f4905,1294
9,client_e547b89c05043229,content_61215c724c8220ae,1285


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot tell us why a content item's search performance changed or prove that one factor caused another. The history is unbalanced, because different clients have different amounts of available search history. Some early rows may also be GSC-only, so data coverage is not equally complete across clients and time periods.

The 90-day window contains shorter 30-day periods (last30 and prev30) that are part of the same overall observation, so these windows can overlap or be related and should not be treated as fully independent observations. The dataset also cannot directly tell us about user search intent, competitor actions, or Google algorithm changes.

Therefore, the data can show patterns and associations in search performance, but it cannot by itself prove causation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.